# Lab | Text Generation from Shakespeare's Sonnet

This notebook explores the fascinating domain of text generation using a deep learning model trained on Shakespeare's sonnets. 

The objective is to create a neural network capable of generating text sequences that mimic the style and language of Shakespeare.

By utilizing a Recurrent Neural Network (RNN) with Long Short-Term Memory (LSTM) layers, this project aims to demonstrate how a model can learn and replicate the complex patterns of early modern English. 

The dataset used consists of Shakespeare's sonnets, which are preprocessed and tokenized to serve as input for the model.

Throughout this notebook, you will see the steps taken to prepare the data, build and train the model, and evaluate its performance in generating text. 

This lab provides a hands-on approach to understanding the intricacies of natural language processing (NLP) and the potential of machine learning in creative text generation.

Let's import necessary libraries

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
import tensorflow.keras.utils as ku
import numpy as np
import matplotlib.pyplot as plt

Let's get the data!

In [ ]:
import requests
url = 'https://raw.githubusercontent.com/martin-gorner/tensorflow-rnn-shakespeare/master/shakespeare/sonnets.txt'
resp = requests.get(url)
with open('sonnets.txt', 'wb') as f:
    f.write(resp.content)

data = open('sonnets.txt').read()

# Split by newline and convert to lowercase
# Each line of a sonnet becomes one sentence in the corpus
corpus = data.lower().split("\n")

print(f"Total lines in corpus: {len(corpus)}")
print("First 5 lines:", corpus[:5])

Step 1: Initialise a tokenizer and fit it on the corpus variable using .fit_on_texts

In [ ]:
# Tokenizer builds a vocabulary mapping every unique word to an integer index
# e.g. {'the': 1, 'and': 2, 'to': 3, 'shall': 4, ...}
# fit_on_texts() scans all sentences in the corpus and learns these mappings
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

print("Vocabulary built!")
print("Sample entries:", {k: v for k, v in list(tokenizer.word_index.items())[:5]})

Step 2: Calculate the Vocabulary Size

In [ ]:
# word_index maps each unique word to an integer starting from 1
# We add 1 because index 0 is reserved for padding (not a real word)
# total_words defines the output layer size: one probability per word
total_words = len(tokenizer.word_index) + 1

print(f"Total unique words in vocabulary: {total_words}")

Create n-gram sequences from the corpus

In [ ]:
input_sequences = []

for line in corpus:
    # Convert the text line into a list of integer token IDs
    # e.g. "shall i compare" -> [4, 2, 67]
    token_list = tokenizer.texts_to_sequences([line])[0]

    # Generate ALL n-gram sub-sequences from this token list
    # For tokens [4, 2, 67, 10] we produce:
    #   [4, 2]         (predict 2 given context 4)
    #   [4, 2, 67]     (predict 67 given context 4, 2)
    #   [4, 2, 67, 10] (predict 10 given context 4, 2, 67)
    # This teaches the model to predict the NEXT word at every position
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)

print(f"Total n-gram sequences created: {len(input_sequences)}")
print("Sample sequences (first 3):")
for seq in input_sequences[:3]:
    print(" ", seq)

Pad sequences to the same length

In [ ]:
# Find the longest sequence — all others will be padded to match this
max_sequence_len = max([len(seq) for seq in input_sequences])
print(f"Maximum sequence length: {max_sequence_len}")

# Pre-padding adds zeros at the BEGINNING of shorter sequences
# e.g. [4, 2] with maxlen=10 -> [0, 0, 0, 0, 0, 0, 0, 0, 4, 2]
# Pre-padding is standard: the real words stay at the end (most recent context)
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

print(f"Input sequences shape: {input_sequences.shape}")

Prepare Predictors and Labels

In [ ]:
# Predictors: all tokens EXCEPT the last -> what the model receives as input
# Labels:     the LAST token only        -> the word the model must predict
# Example: [0, 0, 4, 2, 67] -> predictors=[0, 0, 4, 2], label=67
predictors = input_sequences[:, :-1]   # all columns except last
labels     = input_sequences[:, -1]    # only the last column

print(f"Predictors shape: {predictors.shape}")
print(f"Labels shape:     {labels.shape}")

One-Hot Encode the Labels

In [ ]:
# to_categorical converts integer labels into one-hot encoded vectors
# e.g. label=3 with total_words=10 -> [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
# Needed because the output layer uses softmax: it produces a probability
# distribution over ALL words, and we compare against the one-hot true label
labels = ku.to_categorical(labels, num_classes=total_words)

print(f"Labels shape after one-hot encoding: {labels.shape}")
print(f"Each label is a vector of length {labels.shape[1]} (= total_words)")

# Initialize the Model

In [ ]:
model = Sequential([

    # EMBEDDING LAYER
    # Maps each word index to a dense 100-dimensional vector
    # These vectors are learned during training (similar concept to Word2Vec)
    # input_length = max_sequence_len - 1 because predictors have the last token removed
    Embedding(total_words, 100, input_length=max_sequence_len - 1),

    # BIDIRECTIONAL LSTM (150 units)
    # Runs two LSTMs: one left-to-right, one right-to-left
    # Doubles context: the model sees what came before AND after each word
    # return_sequences=True: passes the full sequence output to the next layer
    Bidirectional(LSTM(150, return_sequences=True)),

    # DROPOUT (rate=0.2)
    # Randomly disables 20% of neurons during training
    # Forces the network to not rely on specific neurons -> better generalization
    Dropout(0.2),

    # LSTM (100 units)
    # Second recurrent layer that condenses the sequence into one final hidden state
    # return_sequences=False (default): only the last output is passed forward
    LSTM(100),

    # DENSE INTERMEDIATE (ReLU + L2 regularization)
    # Half the vocabulary size as intermediate representation
    # L2 penalizes large weights -> reduces overfitting
    Dense(total_words // 2, activation='relu',
          kernel_regularizer=regularizers.l2(0.01)),

    # DENSE OUTPUT (softmax)
    # One unit per word in the vocabulary
    # Softmax converts raw scores into probabilities that sum to 1
    # The word with the highest probability is the predicted next word
    Dense(total_words, activation='softmax')
])

# Compile the Model

In [ ]:
# categorical_crossentropy: standard loss for multi-class classification
# Measures how different the predicted probability distribution is from the true one-hot label
# Adam: adaptive learning rate optimizer, works well for NLP tasks out of the box
# accuracy: fraction of next-word predictions that exactly match the true next word
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Print Model Summary

In [ ]:
# Shows all layers, output shapes, and trainable parameter counts
# Useful to verify the architecture before committing to a long training run
model.summary()

# Train the model for 50 epochs

In [ ]:
# Train on the full dataset for 50 passes (epochs)
# After 50 epochs expect ~40% accuracy (the task is hard: predict exact next word)
# history stores loss and accuracy per epoch for plotting
history = model.fit(
    predictors,
    labels,
    epochs=50,
    verbose=1
)

# Plot training accuracy and loss over epochs

In [ ]:
# history.history is a dict: {'accuracy': [...], 'loss': [...]}
# Each list has one value per epoch
acc  = history.history['accuracy']
loss = history.history['loss']
epochs_range = range(len(acc))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(epochs_range, acc, color='steelblue', linewidth=2)
axes[0].set_title('Training Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(epochs_range, loss, color='tomato', linewidth=2)
axes[1].set_title('Training Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Shakespeare LSTM — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Final Accuracy: {acc[-1]:.4f}")
print(f"Final Loss:     {loss[-1]:.4f}")

# Generate text with the model based on a seed text

In [ ]:
# seed_text: the starting phrase the model uses as context
# next_words: how many new words the model will generate after the seed
seed_text = "shall i compare thee"
next_words = 20

In [ ]:
def generate_text(seed_text, next_words, model, tokenizer, max_sequence_len):
    """
    Generates `next_words` new words given a `seed_text` starting phrase.

    Each iteration:
      1. Tokenize the current text into integer indices
      2. Pre-pad to max_sequence_len - 1 (same shape the model expects)
      3. model.predict() -> probability distribution over all vocabulary words
      4. argmax -> pick the most likely next word index
      5. Convert index back to word and append to the running text
      6. Repeat
    """
    for _ in range(next_words):
        # Step 1: Tokenize the current seed
        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        # Step 2: Pad to the correct input length
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len - 1,
            padding='pre'
        )

        # Step 3: Get probability distribution over vocabulary
        predicted_probs = model.predict(token_list, verbose=0)

        # Step 4: Pick the word with the highest probability (greedy decoding)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        # Step 5: Convert index back to word
        # index_word is the reverse dict: integer -> word
        output_word = tokenizer.index_word.get(predicted_index, '')

        # Step 6: Append to seed and continue
        seed_text += ' ' + output_word

    return seed_text


# Generate and print
generated = generate_text(seed_text, next_words, model, tokenizer, max_sequence_len)
print("Generated text:")
print(generated)

Experiment with at least 3 different seed_text strings and see what happens!

In [ ]:
# --- Experiment 1 ---
# Famous opening line of Sonnet 18
seed1 = "shall i compare thee to a summer"
result1 = generate_text(seed1, 20, model, tokenizer, max_sequence_len)
print("Seed 1 :", seed1)
print("Output :", result1)
print()

# --- Experiment 2 ---
# Opening of Sonnet 29
seed2 = "when in disgrace with fortune"
result2 = generate_text(seed2, 20, model, tokenizer, max_sequence_len)
print("Seed 2 :", seed2)
print("Output :", result2)
print()

# --- Experiment 3 ---
# Opening of Sonnet 116
seed3 = "love is not love which alters"
result3 = generate_text(seed3, 20, model, tokenizer, max_sequence_len)
print("Seed 3 :", seed3)
print("Output :", result3)